# 3. MMELON embeddings + LightGBM

Two steps, the `embed_lgbm` method:
1. **Embed** the training and test molecules with an MMELON encoder. All four views (fused, text, image,
   graph) come from one pass, and the embeddings are saved as parquet in the experiment, so the other views
   and later runs reuse them.
2. **LightGBM** on those embeddings: one model per fold (splits `fold0`, …), one on all training rows, then
   the test file scored by the fold models averaged (`test_ensemble`) and by that model (`test`).

The code is [../methods/embeddings.py](../methods/embeddings.py) and
[../methods/embed_lgbm.py](../methods/embed_lgbm.py). Replaces `MMELON_embedding_extraction`,
`MMELON_embedding_lgbm_cv` and `MMELON_embedding_lgbm_inference` in [../../notebooks/](../../notebooks/).

> With a **fine-tuned** encoder (`ENCODER` set), the encoder has already seen the training labels, so the
> cross-validation numbers are over-optimistic; only the test numbers are meaningful. The pretrained base
> model (`ENCODER = None`) gives a clean cross-validation.

## Settings

In [ ]:
EXPERIMENT = "sample-benchmark"
RUN_NAME = "lgbm_pretrained_fused_notebook"     # folder under the experiment; must not hold a run yet
VIEW = "fused"                                  # "fused", "text", "image" or "graph"
ENCODER = None                                  # None = pretrained base model, or a path to best_model.ckpt
LGBM_PARAMS = "suggested"                       # "suggested" or "default"
TEST_PATH = "../../sample_data/test.parquet"
BATCH_SIZE = 32

In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Make the benchmark code importable (works from this folder or from the repo root)
BENCHMARK = next(p / rel for p in [Path.cwd(), *Path.cwd().parents] for rel in ("", "aircheck/benchmark")
                 if (p / rel / "methods" / "common.py").exists())
sys.path.insert(0, str(BENCHMARK))
AIRCHECK = BENCHMARK.parent
EXP_DIR = AIRCHECK / "output" / EXPERIMENT
print(f"Benchmark code : {BENCHMARK}")
print(f"Experiment     : {EXP_DIR}")

In [ ]:
FOLDS = sorted(EXP_DIR.glob("folds/*_cv_folds.npz"))
assert FOLDS, f"No folds in {EXP_DIR / 'folds'}; run 01_folds.ipynb first (or run_all.sh)"
FOLDS = FOLDS[0]
OUT = EXP_DIR / RUN_NAME
print(f"Folds : {FOLDS.name}")
print(f"Run   : {OUT}")

## 1. Embeddings

`extract` skips the parts it already has, so this is quick the second time and for the other views. The
folder is the one the method itself uses, so the LightGBM step below finds these files.

In [ ]:
from methods import embed_lgbm, embeddings
from run import load_data

data, folds, test = load_data(FOLDS, TEST_PATH)
EMB_DIR = EXP_DIR / "embeddings" / embed_lgbm.encoder_label(ENCODER)
encoder, encoder_view = embeddings.load_encoder(ENCODER)
print(f"encoder: {embed_lgbm.encoder_source(ENCODER)} ({encoder_view}); views {embeddings.available_views(encoder_view)}")

for name, frame in (("train", data), ("test", test)):
    embeddings.extract(encoder, encoder_view, frame["smiles"], EMB_DIR / name,
                       embed_lgbm.encoder_source(ENCODER), batch_size=BATCH_SIZE)

In [ ]:
X_train, valid = embeddings.load(EMB_DIR / "train", VIEW, data["smiles"])
print(f"{X_train.shape[0]:,} training molecules x {X_train.shape[1]} {VIEW} dimensions")
if not valid.all():
    print(f"{(~valid).sum():,} molecules have SMILES RDKit cannot read: no embedding, empty score")
pd.read_parquet(sorted((EMB_DIR / "train" / VIEW).glob("part-*.parquet"))[0]).head(3)

## 2. LightGBM

The embeddings are already there, so this only trains LightGBM (fast) and writes `predictions.csv`,
`run.json` and `models/`.

In [ ]:
from run import run_one

table = run_one("embed_lgbm", folds=FOLDS, test=TEST_PATH, out=OUT,
                settings=dict(view=VIEW, encoder=ENCODER, lgbm_params=LGBM_PARAMS, batch_size=BATCH_SIZE))
table.round(4)

In [ ]:
print(f"LightGBM parameters ({LGBM_PARAMS}): {embed_lgbm.LGBM_PARAMS[LGBM_PARAMS]}")
for path in sorted((OUT / "models").iterdir()):
    print(f"{path.name:16s} {path.stat().st_size / 1e3:.0f} kB")

Score new molecules with these models in [04_inference.ipynb](04_inference.ipynb); build the report
with [05_report.ipynb](05_report.ipynb).